# Blindference Agent SDK — Getting Started

This notebook demonstrates confidential inference with the Blindference Agent SDK.

All prompts are encrypted end-to-end before leaving your machine:
- **AES-256-GCM** encrypts the prompt locally
- **Fhenix CoFHE** distributes the encryption key to quorum nodes
- **1 Leader + N Verifiers** execute the same model and reach consensus
- **On-chain commitment** anchors the accepted result to Arbitrum Sepolia

A development mode (mock) is available for testing without managing keys.

In [ ]:
# Cell 1: Environment Validation
import sys
import shutil
import os
from pathlib import Path

print('Blindference Agent — Environment Check')
print('=' * 50)

checks = []

# Python version
py = sys.version_info
ok = py >= (3, 10)
checks.append(('Python ≥3.10', ok, f'{py.major}.{py.minor}.{py.micro}'))

# Node.js
node = shutil.which('node')
checks.append(('Node.js', bool(node), node or 'not found'))

# npm dependencies
nm = Path('node_modules')
npm_ok = (nm / '@cofhe' / 'sdk').exists() and (nm / 'viem').exists()
checks.append(('npm deps installed', npm_ok, 'run: npm install'))

# .env file
env_ok = Path('.env').exists() or Path('.env.local').exists()
checks.append(('.env file', env_ok, 'create from .env.example'))

# SDK import
try:
    from blindference_agent import BlindferenceAgent
    checks.append(('Agent SDK', True, 'imported'))
except ImportError as e:
    checks.append(('Agent SDK', False, str(e)))

for name, passed, detail in checks:
    status = '✅' if passed else '❌'
    print(f'{status} {name:20s} — {detail}')

all_passed = all(p for _, p, _ in checks)
print()
if all_passed:
    print('Ready for end-to-end encrypted inference.')
else:
    print('Some checks failed. Fix above, or use mock=True for testing.')

## Cell 2: Confidential Inference

Submit an encrypted prompt and receive the decrypted result.

In [ ]:
import asyncio
import os
from blindference_agent import BlindferenceAgent

async def confidential_inference():
    agent = BlindferenceAgent(
        icl_url=os.environ.get('BLF_ICL_URL', 'https://icl.blindference.xyz'),
        cofhe_rpc=os.environ.get('BLF_COFHE_RPC'),
        private_key=os.environ.get('BLF_PRIVATE_KEY'),
    )

    result = await agent.inference(
        prompt='Explain the concept of zero-knowledge proofs in one paragraph',
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
    )

    print('=' * 60)
    print(result.text)
    print('=' * 60)
    print(f'Model:      {result.model_id}')
    print(f'Request:    {result.request_id}')
    print(f'Leader:     {result.leader_address}')
    print(f'Verifiers:  {len(result.verifier_addresses)}')
    print(f'Commitment: {result.commitment_hash[:30]}...')

    await agent.close()

await confidential_inference()

### Development Mode (Testing)

For rapid iteration without managing keys, use `mock=True`. This skips all cryptography and sends plaintext. **Not for production or sensitive data.**

In [ ]:
# DEV MODE — no encryption, no keys required
import asyncio
from blindference_agent import BlindferenceAgent

async def dev_inference():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,
    )

    result = await agent.inference(
        prompt='Explain zero-knowledge proofs',
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
    )

    print(result.text)
    await agent.close()

await dev_inference()

## Cell 3: Streaming Execution Trace

Watch the quorum execute in real time — encryption, assignment, leader inference, verifier replay, on-chain commitment, and decryption.

In [ ]:
import asyncio
from tqdm.notebook import tqdm
from blindference_agent import BlindferenceAgent

async def streaming_trace():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,  # Set mock=False with real keys for encrypted flow
    )

    request = await agent.submit(
        prompt='What is the capital of France?',
        model_id='gemini:gemini-2.5-flash',
        verifier_count=2,
    )

    print(f'Request ID: {request.request_id}')
    print('Watching quorum execution...\n')

    pbar = tqdm(total=5, desc='Execution')
    steps = {
        'quorum': 'Building Quorum',
        'leader': 'Leader Inference',
        'verifier': 'Verifier Replay',
        'onchain': 'On-Chain Commit',
        'decrypt': 'Decrypt Result',
    }

    async for status in agent.stream_status(request.request_id, interval=1.0):
        pbar.set_description(steps.get(status.step, status.step))
        if status.status in ('ASSIGNED', 'EXECUTING', 'VERIFYING'):
            pbar.update(1)
        elif status.status == 'ACCEPTED':
            pbar.update(2)
            pbar.close()
            print('\n✅ Quorum consensus reached')
            break
        elif status.status in ('REJECTED', 'DISPUTED'):
            pbar.close()
            print(f'\n❌ Failed: {status.status}')
            break

    await agent.close()

await streaming_trace()

## Cell 4: Batch Inference

Submit multiple prompts concurrently. Each runs through its own quorum.

In [ ]:
import asyncio
from blindference_agent import BlindferenceAgent

async def batch_inference():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,  # Set mock=False with real keys
    )

    prompts = [
        'What is machine learning?',
        'Explain neural networks briefly',
        'What is overfitting?',
    ]

    print(f'Submitting {len(prompts)} prompts concurrently...\n')

    requests = await asyncio.gather(*[
        agent.submit(prompt, 'groq:llama-3.3-70b-versatile', verifier_count=2)
        for prompt in prompts
    ])

    results = []
    for req in requests:
        while True:
            status = await agent.icl.get_status(req.request_id)
            if status.status == 'ACCEPTED':
                result = await agent._decrypt_result(status, req.model_id)
                results.append(result)
                break
            await asyncio.sleep(1.0)

    print('=' * 60)
    for prompt, result in zip(prompts, results):
        print(f'\nQ: {prompt}')
        print(f'A: {result.text[:100]}...')
        print(f'   Request: {result.request_id}')
    print('=' * 60)

    await agent.close()

await batch_inference()

## Cell 5: Result Metadata

Inspect the full `InferenceResult` — on-chain commitment, quorum composition, timestamps.

In [ ]:
import asyncio
from blindference_agent import BlindferenceAgent

async def inspect_metadata():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,  # Set mock=False with real keys
    )

    result = await agent.inference(
        prompt='What is the Pythagorean theorem?',
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
    )

    print('InferenceResult Metadata')
    print('-' * 50)
    for field, value in result.__dict__.items():
        display = value
        if isinstance(value, str) and len(value) > 40:
            display = value[:40] + '...'
        print(f'  {field:25s}: {display}')

    print()
    print(f'Text length:   {len(result.text)} chars')
    print(f'Verifier count: {len(result.verifier_addresses)}')

    await agent.close()

await inspect_metadata()

## Cell 6: Quorum Preview

See which nodes will execute your request before submitting.

In [ ]:
import asyncio
from blindference_agent import BlindferenceAgent

async def preview_quorum():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,
    )

    quorum = await agent.icl.quorum_preview(
        model_id='groq:llama-3.3-70b-versatile',
        verifier_count=2,
        min_tier=0,
    )

    print('Selected Quorum')
    print('-' * 50)
    print(f'Leader:    {quorum.leader.address} (tier {quorum.leader.tier}, rep {quorum.leader.reputation_score})')
    for i, v in enumerate(quorum.verifiers, 1):
        print(f'Verifier {i}: {v.address} (tier {v.tier}, rep {v.reputation_score})')

    await agent.close()

await preview_quorum()

## Cell 7: Interactive Session

A simple chat loop with the Blindference quorum.

In [ ]:
import asyncio
from blindference_agent import BlindferenceAgent

async def interactive_session():
    agent = BlindferenceAgent(
        icl_url='https://icl.blindference.xyz',
        mock=True,  # Set mock=False with real keys for encrypted chat
    )

    print('Blindference Agent — Interactive Session')
    print("Type 'exit' to quit\n")
    print('=' * 60)

    while True:
        user_input = input('You: ')
        if user_input.lower() in ('exit', 'quit', 'q'):
            break

        print('  Processing...')
        result = await agent.inference(
            prompt=user_input,
            model_id='groq:llama-3.3-70b-versatile',
            verifier_count=2,
        )

        print(f'Agent: {result.text}')
        print(f'  [Leader: {result.leader_address[:20]}..., Verifiers: {len(result.verifier_addresses)}]\n')

    print('Session ended.')
    await agent.close()

await interactive_session()